# 03 — The optimal categorical split

*Chapter 10 · LightGBM · Notebook 3 of 5*

A decision tree splits a *numeric* feature with a threshold: `x < t`. But what about a **categorical**
feature — colour, city, product id — whose values have no natural order? Splitting it means choosing
**which categories go left and which go right**. With `K` categories there are `2^(K−1) − 1` ways to do
that, and that grows explosively. This notebook builds the trick that makes it tractable and **exact**:
sort the categories by one number, then check only `K − 1` splits.

You have used native categorical handling once already — Chapter 09 mentioned XGBoost accepts categorical
columns — but we never built the split itself. Here we do, by hand, and match LightGBM's categorical
split exactly.

**Prerequisites.** Chapter 09 NB 2 (the structure-score gain `½G²/(H+λ)`); Chapters 08–09 (the per-row
gradient `g`); this chapter's NB 1–2 (leaf-wise growth; the gradient as a row's signal).

**What you'll be able to do.**
- Count the partitions of a categorical feature and see why brute force is hopeless.
- Summarise each category by one number — its gradient statistic `G/H` — and sort by it.
- Build the optimal split as the best of `K − 1` contiguous cuts, and confirm by brute force that it is
  the global optimum.
- Match the by-hand split to LightGBM's, exactly.

## The problem: too many ways to split a categorical feature

A numeric split is a threshold — one cut on an ordered line. A categorical feature has no line: to split
`K` categories into a left group and a right group, you choose a subset for the left. There are `2^K`
subsets; dropping the empty/full ones and counting each split once (left/right are interchangeable)
leaves **`2^(K−1) − 1`** distinct partitions:

- `K = 6` categories → `31` partitions (we can list them),
- `K = 30` categories → over **500 million**.

Trying them all is hopeless past a handful of categories. The naive workarounds — one-hot encoding (one
yes/no feature per category, which forbids grouping categories together) or pretending the codes are
ordered — either explode the feature count or impose an order that is not there. We want the **best**
grouping, found cheaply.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0
rng = np.random.default_rng(SEED)

# One categorical feature with K = 6 categories. Each category has its own mean
# target, and the means are in a DELIBERATELY scrambled order -- the category index
# (0..5) carries no order, exactly like red/blue/green or a list of cities.
K = 6
means = {0: 2.0, 1: 5.0, 2: 1.0, 3: 4.0, 4: 0.0, 5: 3.0}
per = 120
codes = np.repeat(np.arange(K), per)      # the category of each row
y = np.array([means[c] for c in codes]) + rng.normal(0, 0.05, size=K * per)
n = len(y)

F0 = float(y.mean())     # the constant first prediction (ch 08)
g = F0 - y               # gradient of 1/2 (y - F)^2 at F0
h = np.ones(n)           # squared-error curvature
LAMBDA = 0.0             # ch 09 NB 2's lambda, turned off

print(f"n = {n}  ({per} rows x K = {K} categories);  F0 = mean(y) = {F0:.4f}")
for c in range(K):
    print(f"  category {c}: mean(y) = {y[codes == c].mean():.3f}")


## One number per category: the gradient statistic `G/H`

Chapter 09 NB 2 summarised any group of rows by two sums — the gradient sum `G = Σg` and the curvature
sum `H = Σh` — and gave the group's best leaf value as `−G/H`. So we can summarise **each category** the
same way: collect its rows, add up their `g` and `h`, and read off `G/H`.

For this regression toy the curvature is `h = 1`, so a category's `H` is its row count and its

`G/H = (Σ g) / (count) = mean of g in the category = F₀ − mean(y_c)`.

In words: with squared error, sorting categories by `G/H` is sorting them by their **mean target**. We
will use `G/H` rather than the raw mean, because `G/H` is the general key — it is exactly what carries
over to classification, where `h = p(1−p)` is not `1` and the mean is no longer the right summary.

In [ ]:
catG = np.array([g[codes == c].sum() for c in range(K)])
catH = np.array([h[codes == c].sum() for c in range(K)])
catRatio = catG / catH        # the gradient statistic G/H (= F0 - mean(y_c) here)

print(f"{'cat':>4} {'G':>10} {'H':>6} {'G/H':>9}")
for c in range(K):
    print(f"{c:>4} {catG[c]:>10.2f} {catH[c]:>6.0f} {catRatio[c]:>9.4f}")

order = np.argsort(catRatio)      # categories sorted by their gradient statistic
print(f"\ncategories sorted by G/H (low -> high): {order.tolist()}")


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.bar(np.arange(K), catRatio, color=COLORS["class_d"], edgecolor=COLORS["text"], linewidth=0.5)
ax.axhline(0, color=COLORS["muted"], linewidth=1)
ax.set_xlabel("category index (no natural order)")
ax.set_ylabel("gradient statistic  G/H")
ax.set_xticks(np.arange(K))
ax.set_title("each category becomes one number: its G/H")
plt.show()


**Read the figure.** Each bar is one category's `G/H`. The categories sit in their arbitrary index
order (0,1,2,…), and the heights jump around — category 4 is highest, category 1 lowest. `G/H` has turned
an unordered feature into points on a line. The next step is to **order** the categories by that line.

## Fisher's result: sort, then the best split is contiguous

Here is the idea that rescues us. Sort the categories by `G/H`. Then **the optimal two-group partition
never interleaves** the sorted order — the best split is one of the `K − 1` **contiguous** cuts:
`{first 1} | rest`, `{first 2} | rest`, …, `{first K−1} | rest`.

Why (the intuition, not the full proof): the gain we maximise is a **sum of convex functions**
`½G²/(H+λ)` of the two group totals. The maximum of such a sum is reached at an *extreme* arrangement of
the sorted categories — one where no high-`G/H` category sits on the same side as a lower-`G/H` one across
the boundary — which is exactly a contiguous cut. Fisher (1958) proved this for least-squares grouping
(sort by the group **mean**); the same argument extends to any convex split criterion, which is how
Breiman's CART and LightGBM use it. We will not lean on the sketch alone — in a moment we **brute-force
every partition** and check that the contiguous winner really is the global best.

The consequence is dramatic: we go from `2^(K−1) − 1` partitions to **`K − 1`** — exponential to linear.

In [ ]:
def gain(left_cats):
    """Half the structure-score gain (ch 09 NB 2, lambda = LAMBDA) for putting `left_cats` left."""
    left = np.isin(codes, list(left_cats))
    GL, HL = g[left].sum(), h[left].sum()
    GR, HR = g[~left].sum(), h[~left].sum()
    G, H = g.sum(), h.sum()
    if HL <= 0 or HR <= 0:
        return -np.inf
    return 0.5 * (GL**2 / (HL + LAMBDA) + GR**2 / (HR + LAMBDA) - G**2 / (H + LAMBDA))


# Fisher: walk the K-1 contiguous cuts of the G/H-sorted order.
contig_gains = []
for j in range(1, K):
    left = set(order[:j].tolist())
    contig_gains.append(gain(left))
    print(f"  cut after {j}: left = {sorted(left)}   gain = {contig_gains[-1]:.2f}")

best_j = int(np.argmax(contig_gains)) + 1
best_left = set(order[:best_j].tolist())
print(f"\nbest contiguous split: left = {sorted(best_left)}   gain = {max(contig_gains):.2f}")


In [ ]:
cuts = np.arange(1, K)
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(cuts, contig_gains, "o-", color=COLORS["model"], linewidth=2.2)
ax.plot(best_j, max(contig_gains), "o", color=COLORS["highlight"], markersize=15,
        zorder=5, label=f"best: left = {sorted(best_left)}")
ax.set_xlabel("cut position in the G/H-sorted order")
ax.set_ylabel("structure-score gain")
ax.set_xticks(cuts)
ax.set_title(f"only K-1 = {K - 1} cuts to check, not 2^(K-1)-1 = {2 ** (K - 1) - 1}")
ax.legend()
plt.show()


**Read the figure.** Walking left to right through the `K − 1 = 5` contiguous cuts, the gain climbs
to a single peak and falls. The peak is the cut that separates the high-`G/H` categories from the
low-`G/H` ones — the best grouping. One pass over five cuts found it; we never enumerated the 31
partitions.

## Is the contiguous winner really the best of *all* partitions?

A fair doubt: maybe some clever **non-contiguous** grouping — one that interleaves the sorted order —
scores higher, and Fisher's shortcut quietly misses it. With `K = 6` we can settle it honestly: there are
only `31` partitions, so we enumerate **every** one and compare its gain to our contiguous winner.

In [ ]:
best_brute_gain, best_brute_left, n_part = -np.inf, None, 0
cats = list(range(K))
for size in range(1, K):
    for left in itertools.combinations(cats, size):
        if 0 not in left:           # fix category 0 on the left -> count each partition once
            continue
        n_part += 1
        gn = gain(set(left))
        if gn > best_brute_gain:
            best_brute_gain, best_brute_left = gn, set(left)

print(f"enumerated {n_part} distinct partitions  (= 2^(K-1)-1 = {2 ** (K - 1) - 1})")
print(f"global best: left = {sorted(best_brute_left)}   gain = {best_brute_gain:.2f}")
same = best_brute_left in (best_left, set(cats) - best_left)
print(f"contiguous optimum == global optimum?  {same}")
print()
for kk in [6, 10, 30]:
    print(f"  K = {kk:2d}:  contiguous {kk - 1:>2} cuts   vs   brute force {2 ** (kk - 1) - 1:,}")


**The payoff.** The brute-force global optimum is the partition `{1, 3, 5} | {0, 2, 4}` — the
**same** one our contiguous scan found (our scan names `{1, 3, 5}` the left side; the brute force, which
pins category 0 to the left, reports the complement `{0, 2, 4}` — the identical split). So the contiguous
cut *is* the global best here: Fisher's sort turned an exponential search into a linear one with **no loss
of optimality** (under the two conditions we spell out at the end). At `K = 6` it is 5 cuts instead of 31;
at `K = 30` it is 29 cuts instead of over half a billion. This is why LightGBM can split a high-cardinality
categorical feature natively and fast, instead of forcing you to one-hot it.

## Matching LightGBM

LightGBM does exactly this: for a categorical feature it sorts the categories by their gradient statistic
and takes the best contiguous split (Ke et al. 2017 §4, building on Fisher). To see the bare mechanism we
turn off the guards LightGBM normally applies to categoricals — the minimum rows per group
(`min_data_per_group`), the category-split regularizers (`cat_l2`, `cat_smooth`), and the leaf-size floors
— and grow a single two-leaf tree. Its split should be our partition, exactly.

(We leave LightGBM's log visible — including its warnings that our floor settings override the defaults;
that is the engine reporting what we asked of it.)

In [ ]:
df = pd.DataFrame({"cat": pd.Categorical(codes)})
lgbm = LGBMRegressor(
    n_estimators=1, num_leaves=2, learning_rate=1.0,
    min_data_per_group=1, min_data_in_leaf=1, min_sum_hessian_in_leaf=0.0,
    cat_l2=0.0, cat_smooth=0.0, min_split_gain=0.0, reg_lambda=0.0,
    max_bin=100000, min_data_in_bin=1, random_state=SEED,
)
lgbm.fit(df, y, categorical_feature=["cat"])

# Recover LightGBM's partition: a one-tree, two-leaf model gives each category one of
# two predictions -> group the categories by the leaf value they land in.
probe = pd.DataFrame({"cat": pd.Categorical(np.arange(K), categories=np.arange(K))})
leaf_val = np.round(lgbm.predict(probe), 6)
groups = {}
for c, v in enumerate(leaf_val):
    groups.setdefault(v, []).append(c)
lgbm_partition = sorted([sorted(s) for s in groups.values()])

byhand_partition = sorted([sorted(best_left), sorted(set(cats) - best_left)])
print(f"LightGBM single-tree partition: {lgbm_partition}")
print(f"by-hand partition:              {byhand_partition}")
print(f"identical?  {lgbm_partition == byhand_partition}")


In [ ]:
from matplotlib.patches import Patch

side_color = [COLORS["class_a"] if c in best_left else COLORS["class_b"] for c in range(K)]
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.bar(np.arange(K), leaf_val, color=side_color, edgecolor=COLORS["text"], linewidth=0.5)
ax.set_xlabel("category")
ax.set_ylabel("LightGBM leaf value")
ax.set_xticks(np.arange(K))
ax.legend(handles=[
    Patch(facecolor=COLORS["class_a"], edgecolor=COLORS["text"],
          label=f"by-hand left  {sorted(best_left)}"),
    Patch(facecolor=COLORS["class_b"], edgecolor=COLORS["text"],
          label=f"by-hand right {sorted(set(cats) - best_left)}"),
])
ax.set_title("LightGBM's two leaf values == the by-hand partition")
plt.show()


**Read the figure.** LightGBM gives every category one of **two** leaf values — that is the
partition. The bars are coloured by our **by-hand** side, and the colour matches the height perfectly:
the categories we put on the left all get the high leaf value, the rest get the low one. (Recall
`g = F₀ − y`, so a high-mean category has a *negative* `G/H` and sorts to the low end, while its leaf
value `−G/H` turns positive — the high bar.) The by-hand Fisher sort *is* what LightGBM does inside.

## Scope and honesty

This exactness rests on two conditions. First, a **convex** split criterion — the structure-score gain
qualifies. Second, a **one-dimensional** target summary: regression (the mean), or **binary**
classification (where `G/H` uses `h = p(1−p)` and still gives one number per category). For **multiclass**
classification the optimality breaks — each category carries a *vector* of per-class gradients, and a
single `G/H` ordering is no longer provably optimal; LightGBM falls back to a heuristic there.

Two more honest notes. Our toy is small and almost noiseless, so the partition is crisp; on real data the
guards we switched off — `min_data_per_group`, `cat_l2`, `cat_smooth` — earn their keep, stopping a rare
category (few rows, a wild `G/H`) from carving out its own branch and overfitting. And native categorical
handling is a convenience and a speed win, not a guarantee of higher accuracy — a well-encoded feature can
do as well.

## Your turn

1. **(easy)** Change the category means in the setup (give them a different scrambled order), re-run, and
   read off the new `G/H` sort and the new best cut. Does the winning partition still separate high-mean
   from low-mean categories?
2. **(core)** Print `2^(K−1) − 1` and `K − 1` side by side for `K = 4, 8, 16, 32`. At what `K` does brute
   force stop being feasible, and how many cuts does Fisher's sort check there?
3. **(reach)** Add a 7th category with only a handful of rows and an extreme mean. Re-fit LightGBM with
   `min_data_per_group=1` and then with the default `100`: how does the split change? Explain what the
   floor is protecting against.

## What you built

You built the **optimal categorical split** by hand: summarise each category by its gradient statistic
`G/H`, sort, and take the best of the `K − 1` contiguous cuts. You confirmed by brute force that this
contiguous winner is the **global** optimum among all `2^(K−1) − 1` partitions, and matched it to
LightGBM's categorical split exactly. The lesson is Fisher's: sorting by the right one-number summary
collapses an exponential search to a linear one, with no loss.

**Vocabulary.** categorical split · gradient statistic `G/H` · contiguous partition · `2^(K−1)−1` → `K−1`
· `min_data_per_group` / `cat_l2` / `cat_smooth` (the categorical guards).

Next: **the estimator `LGBMClassifier` / `LGBMRegressor` and its parameters** — where `num_leaves`,
`min_child_samples`, the learning rate, GOSS, native categoricals, and early stopping come together.

## References

- Fisher, W. D. (1958). *On grouping for maximum homogeneity.* Journal of the American Statistical
  Association 53(284), 789-798. DOI
  [10.1080/01621459.1958.10501479](https://doi.org/10.1080/01621459.1958.10501479). (The optimal
  partition is contiguous once groups are sorted by their mean — the result this notebook builds.)
- Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision
  Tree.* Advances in Neural Information Processing Systems 30 (NeurIPS 2017). (§4 — native categorical
  handling via the sorted-gradient split.)
- Breiman, L., Friedman, J., Olshen, R., & Stone, C. (1984). *Classification and Regression Trees.*
  Wadsworth. (The same sort-by-mean optimal categorical split for a binary target.)
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). (The structure-score gain reused
  here — Chapter 09 NB 2.)

*Previous: 02 — GOSS and EFB.*  ·  *Next: 04 — The estimator and its parameters.*